This notebook fulfills the remaining requirements:

- Requirement 5 & Bonus 5: Unsupervised Learning (K-Means Clustering).

- Requirement 5 & 6: Supervised Learning (Logistic Regression / Random Forest).

- Requirement 7: Model Evaluation (Precision, Recall, Accuracy).

## Data Preparation & Preprocessing
To ensure our Machine Learning models treat all variables equally (e.g., so that €100M spending doesn't "overwhelm" 50 points), we must scale our data.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Environment & Data Loading
load_dotenv()
engine = create_engine(f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}")

query = """
SELECT 
    l.name as league, p.season_year, t.name as team, p.points, p.is_top_5,
    p.goals_for, p.goals_against, (p.goals_for - p.goals_against) as goal_diff,
    sh.stadium_capacity, e.transfer_spend_euro, e.net_transfer_spend_euro
FROM team_season_performance p
JOIN teams t ON p.team_id = t.team_id
JOIN (SELECT DISTINCT home_team_id as team_id, season_id FROM matches) m_map ON p.team_id = m_map.team_id
JOIN seasons s ON m_map.season_id = s.season_id AND p.season_year = s.year
JOIN leagues l ON s.league_id = l.league_id
JOIN team_stadium_history sh ON p.team_id = sh.team_id AND p.season_year = sh.season_year
JOIN team_enrichment_data e ON p.team_id = e.team_id AND p.season_year = e.season_year;
"""

df = pd.read_sql(query, engine)

# Re-apply our creative feature
df['transfer_spend_euro'] = df['transfer_spend_euro'].replace(0, 1)
df['cost_per_goal_m_euro'] = (df['transfer_spend_euro'] / df['goals_for']) / 1_000_000

print(f"Data successfully re-loaded: {df.shape[0]} records.")

Data successfully re-loaded: 290 records.


## Unsupervised Learning - K-Means Clustering
We group teams by their "Strategic Profile." We use scaling to ensure that large Euro values don't drown out point values.

# Select features for profiling
cluster_features = ['stadium_capacity', 'transfer_spend_euro', 'cost_per_goal_m_euro', 'points']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[cluster_features])

# Apply K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Visualize the result
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='transfer_spend_euro', y='points', hue='cluster', palette='viridis', s=100)
plt.title('K-Means Clustering: European Club Profiles')
plt.show()